In [ ]:
import os
import sys
from typing import Any, List
import pdfplumber
import polars as pl
from pydantic import BaseModel, field_validator
from IPython.display import display

# Add parent directory to Python path
notebook_dir = os.path.dirname(os.path.abspath(''))
parent_dir = os.path.dirname(notebook_dir)
if parent_dir not in sys.path:
	sys.path.append(parent_dir)

from src.utils.converter import format_luas, format_number, format_text, normalize_kabupaten_kota, normalize_kecamatan, normalize_kelurahan_desa

In [ ]:
current_path = os.getcwd()
script_dir = os.path.dirname(current_path)
input_dir = os.path.join(script_dir, "datas")
print(current_path)
print(input_dir)

In [ ]:
file_name = "Keputusan_Menteri_Dalam_Negeri_Nomor_300.2.2-2138_Tahun_2025.pdf"
pdf_path = os.path.join(input_dir, file_name)

In [ ]:
detail_settings = {
    # "vertical_strategy": "lines",
    # "horizontal_strategy": "lines",
    # "snap_y_tolerance": 7,
    # "intersection_y_tolerance": 500,
    # "intersection_x_tolerance": 500,
}

In [ ]:
start_page_index = 619
end_page_index = 626

page_numbers = list(range(start_page_index, end_page_index + 1))
print(page_numbers)

In [ ]:
with pdfplumber.open(pdf_path, pages=page_numbers) as pdf:
    if pdf.pages:
        for i, page in enumerate(pdf.pages):
            print(f"\nProcessing page {i + 1}:")
            table = page.find_table()
            if table is not None and table.bbox is not None:
                table_bbox = table.bbox
                the_table = page.crop(table_bbox)
                im = the_table.to_image(resolution=150)
                table_extract = im.debug_tablefinder(detail_settings)
                value_table = the_table.extract_table(detail_settings)
                display(table_extract)
                replacement_value = ""

In [ ]:
raw_records = []

with pdfplumber.open(pdf_path, pages=page_numbers) as pdf:
    if pdf.pages:

        for i, page in enumerate(pdf.pages):
            table = page.extract_table(detail_settings)
            if table is not None:
                for row in table:
                    if any(row) :
                        raw_records.append(row)

In [ ]:
def are_sublist_lengths_same(main_list):
    if not main_list:
        return True

    lengths = {len(sublist) for sublist in main_list}
    if len(lengths) == 1:
        return "Ok"
    else:
        raise ValueError("Not same length")


In [ ]:
print(are_sublist_lengths_same(raw_records))
print(len(raw_records[0]))

In [ ]:
raw_records

In [ ]:
class DetailsData(BaseModel):
    kode_provinsi: str
    provinsi: str
    jumlah_kabupaten: int
    jumlah_kota: int
    kode_kabupaten_kota: str
    kabupaten_kota: str
    kode_kecamatan: str
    kecamatan: str
    kode_kelurahan: str
    kelurahan: str
    desa: str
    luas_wilayah_km2: float
    keterangan: str

    # Field validators for data type conversion and validation
    @field_validator("jumlah_kabupaten", "jumlah_kota", mode="before")
    @classmethod
    def validate_int_fields(cls, v):
        if v is None or v == "":
            return 0
        return int(format_number(v))

    @field_validator("luas_wilayah_km2", mode="before")
    @classmethod
    def validate_float_fields(cls, v):
        if v is None or v == "":
            return 0.0
        return float(format_luas(v))

    @field_validator("kode_provinsi", "provinsi",
                     "kode_kabupaten_kota", "kabupaten_kota",
                     "kode_kecamatan", "kecamatan", "kode_kelurahan", "kelurahan", "desa",
                     "keterangan", mode="before")
    @classmethod
    def validate_str_fields(cls, v):
        """Normalize all string fields to clean newlines and extra whitespace."""
        return format_text(v)
    
    @field_validator("kabupaten_kota", mode="before")
    @classmethod
    def validate_kabupaten_kota_field(cls, v):
        """Normalize kabupaten_kota field by expanding abbreviations."""
        return normalize_kabupaten_kota(v)
    
    @field_validator("kecamatan", mode="before")
    @classmethod
    def validate_kecamatan_field(cls, v):
        """Normalize kecamatan/kelurahan/desa field by expanding abbreviations."""
        return normalize_kecamatan(v)
    
    # @field_validator("kelurahan", "desa", mode="before")
    # @classmethod
    # def validate_kelurahan_desa_field(cls, v):
    #     """Normalize kelurahan/desa field by expanding abbreviations."""
    #     return normalize_kelurahan_desa(v)

In [ ]:
clean_records = []

# Context variables for hierarchical data
current_province = {
    'kode_provinsi': '',
    'provinsi': '',
    'jumlah_kabupaten': 0,
    'jumlah_kota': 0
}

current_regency = {
    'kabupaten_kota': '',
    'jumlah_kecamatan': 0,
    'jumlah_kelurahan': 0,
    'jumlah_desa': 0
}

current_district = {
    'kecamatan': '',
    'kelurahan': '',
    'desa': ''
}

# Helper for safe row access (avoid shadowing)
def safe_row_val(row_data: List[Any], idx: int, default: Any = "") -> Any:
    return row_data[idx] if len(row_data) > idx and row_data[idx] is not None else default


for row in raw_records:
    if not row or len(row) < 9:
        continue

    kode = safe_row_val(row, 0, "")
    prov_kab = safe_row_val(row, 1, "")
    jumlah_kab = safe_row_val(row, 2, "")
    jumlah_kota = safe_row_val(row, 3, "")
    keterangan = safe_row_val(row, 8, "")

    if kode == "" and prov_kab and not prov_kab.isdigit() and prov_kab not in ['KAB', 'KOTA', 'KEC']:
        # Only process real historical districts with meaningful names
        data = DetailsData(
            kode_provinsi=current_province['kode_provinsi'],
            provinsi=current_province['provinsi'],
            jumlah_kabupaten=0,
            jumlah_kota=0,
            kode_kabupaten_kota="",
            kabupaten_kota="",
            kode_kecamatan="",
            kecamatan="",
            kode_kelurahan="",
            kelurahan="",
            desa="",
            luas_wilayah_km2=0.0,
            keterangan=keterangan
        )
        clean_records.append(data)
        continue

    # Skip headers
    if kode in ['K O D E', None] or prov_kab in ['NAMA PROVINSI /\nKABUPATEN / KOTA', None] or jumlah_kab == 'KAB':
        continue

    # Parse codes
    kode_parts = kode.split('.') if kode else []
    if len(kode_parts) < 1:
        continue

    kode_provinsi = kode_parts[0]
    kode_kabupaten_kota = f"{kode_parts[0]}.{kode_parts[1]}" if len(kode_parts) > 1 else ""
    kode_kecamatan = f"{kode_parts[0]}.{kode_parts[1]}.{kode_parts[2]}" if len(kode_parts) > 2 else ""
    kode_kelurahan = kode if len(kode_parts) == 4 else ""

    if len(kode_parts) == 1:  # Province
        # Update context FIRST
        current_province.update({
            'kode_provinsi': kode_provinsi,
            'provinsi': prov_kab,
            'jumlah_kabupaten': int(jumlah_kab) if jumlah_kab.isdigit() else 0,
            'jumlah_kota': int(jumlah_kota) if jumlah_kota.isdigit() else 0
        })

        data = DetailsData(
            kode_provinsi=kode_provinsi,
            provinsi=prov_kab,
            jumlah_kabupaten=safe_row_val(row, 2, ""),
            jumlah_kota=safe_row_val(row, 3, ""),
            kode_kabupaten_kota="",
            kabupaten_kota="",
            kode_kecamatan="",
            kecamatan="",
            kode_kelurahan="",
            kelurahan="",
            desa="",
            luas_wilayah_km2=safe_row_val(row, 7, ""),
            keterangan=keterangan
        )
        clean_records.append(data)

    elif len(kode_parts) == 2:  # Regency
        current_regency.update({
            'kabupaten_kota': prov_kab
        })

        base_data = {
            'kode_provinsi': kode_provinsi,
            'provinsi': current_province['provinsi'],
            'luas_wilayah_km2': safe_row_val(row, 7, ""),
            'keterangan': keterangan
        }

        data = DetailsData(
            **base_data,
            jumlah_kabupaten=0,
            jumlah_kota=0,
            kode_kabupaten_kota=kode_kabupaten_kota,
            kabupaten_kota=prov_kab,
            kode_kecamatan="",
            kecamatan="",
            kode_kelurahan="",
            kelurahan="",
            desa="",
        )
        clean_records.append(data)

    elif len(kode_parts) == 3:  # District
        current_district.update({
            'kecamatan': safe_row_val(row, 4, ""),
            'kelurahan': safe_row_val(row, 5, ""),
            'desa': safe_row_val(row, 6, "")
        })

        base_data = {
            'kode_provinsi': kode_provinsi,
            'provinsi': current_province['provinsi'],
            'luas_wilayah_km2': 0.0,  # Not in district data
            'keterangan': keterangan
        }

        data = DetailsData(
            **base_data,
            jumlah_kabupaten=0,
            jumlah_kota=0,
            kode_kabupaten_kota=kode_kabupaten_kota,
            kabupaten_kota=current_regency['kabupaten_kota'],
            kode_kecamatan=kode_kecamatan,
            kecamatan=safe_row_val(row, 4, ""),
            kode_kelurahan="",
            kelurahan="",
            desa=""
        )
        clean_records.append(data)

    elif len(kode_parts) == 4:  # Sub-district or Village
        subdistrict_name = safe_row_val(row, 5, "")
        village_name = safe_row_val(row, 6, "") 

        data = DetailsData(
            kode_provinsi=kode_provinsi,
            provinsi=current_province['provinsi'],
            jumlah_kabupaten=0,
            jumlah_kota=0,
            kode_kabupaten_kota=kode_kabupaten_kota,
            kabupaten_kota=current_regency['kabupaten_kota'],
            kode_kecamatan=kode_kecamatan,
            kecamatan=current_district['kecamatan'],
            kode_kelurahan=kode_kelurahan,
            kelurahan=subdistrict_name,
            desa=village_name,
            luas_wilayah_km2=0.0,  # Not in village data
            keterangan=keterangan
        )
        clean_records.append(data)
        
clean_records

In [ ]:
df = pl.DataFrame([data.model_dump() for data in clean_records])

In [ ]:
# Create cleaned version with clean names AND remove moved subdistricts/villages
df_clean = df.with_columns([
    # Clean kelurahan names (remove numbering)
    pl.col("kelurahan").str.replace(r'^\d+\s+', '', literal=False).alias("kelurahan"),
    # Clean desa names (remove numbering)
    pl.col("desa").str.replace(r'^\d+\s+', '', literal=False).alias("desa")
]).filter(
    # Remove moved districts (those with empty subdistrict/village)
    pl.col("kode_kelurahan") != ""
)

In [ ]:
df

In [ ]:
df_clean